In [16]:
import numpy as np
import pandas as pd
import requests

In [17]:
BASE_URL = "https://leaderboard-project1-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "/home/renku/work/input-data/Project 1 - DTB/bioreaction_template.csv"
SUBMISSION_PATH = "submissions/hybrid_ensemble.csv"   # predictions to submit (same rows as the template)
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    submission = {"predictions": df.to_dict(orient="records")}
    response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    return response.json()["metrics"]

In [18]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 7,1,1,2026-11-21T18:06:56.327299,6.745,6.649,7.234,7.446,7.018
1,Team 04,2,4,2026-11-21T16:16:43.084759,7.151,6.903,7.546,6.969,7.142
2,Team 15,3,3,2026-11-21T15:33:11.666159,6.617,6.515,7.949,8.292,7.343
3,Team 01,4,1,2026-11-21T17:04:14.729206,6.975,6.802,8.615,8.551,7.736
4,Team 10,5,57,2026-11-21T18:38:08.031411,7.233,7.214,8.284,8.241,7.743
5,Team 23,6,9,2026-11-21T15:52:24.579771,7.269,7.144,8.346,9.282,8.010
6,Team 18,7,7,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
7,TEMPLATE,8,31,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517


In [19]:
template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
df = pd.read_csv(SUBMISSION_PATH, dtype={"batch_id": str})
# Sanity checks: same rows / order as the template, no missing or negative values
assert (df[["batch_id", "elapsed_time"]].values == template[["batch_id", "elapsed_time"]].values).all()
assert df[TARGETS].notna().all().all() and (df[TARGETS] > 0).all().all()
metrics = submit(df[["batch_id", "elapsed_time", *TARGETS]], user="Team 10")
metrics

{'vcd_mape': 7.279,
 'tcd_mape': 7.245,
 'substrate_mape': 8.454,
 'byproduct_mape': 8.772,
 'mean_mape': 7.938}

In [20]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 7,1,1,2026-11-21T18:06:56.327299,6.745,6.649,7.234,7.446,7.018
1,Team 04,2,4,2026-11-21T16:16:43.084759,7.151,6.903,7.546,6.969,7.142
2,Team 15,3,3,2026-11-21T15:33:11.666159,6.617,6.515,7.949,8.292,7.343
3,Team 01,4,1,2026-11-21T17:04:14.729206,6.975,6.802,8.615,8.551,7.736
4,Team 10,5,58,2026-11-21T18:38:08.031411,7.233,7.214,8.284,8.241,7.743
5,Team 23,6,9,2026-11-21T15:52:24.579771,7.269,7.144,8.346,9.282,8.010
6,Team 18,7,7,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
7,TEMPLATE,8,31,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517
